### V0.4

* 1) age      :int64	    顧客の年齢。 
* 2) job      :object	職業（admin, blue-collar, technician など）。　
* 3) marital  :object	既婚・未婚などの婚姻状況。　
* 4) education:object	学歴（primary, secondary, tertiary など）
                    ※primary: 初等教育（小学校など）
                     secondary: 中等教育（中学校・高校など）
                     tertiary: 高等教育（大学・大学院など）
                     unknown: 不明
* 5) default  :object	債務不履行があるかどうか（yes / no）　過去にローンやクレジットカードの支払いが滞ったことがあるか
* 6) balance  :int64	    年間の平均残高。
* 7) housing  :object	住宅ローンがあるかどうか（yes / no）。
* 8) loan     :object	個人ローンがあるかどうか（yes / no）
* 9) contact  :object	連絡手段（cellular（スマホ）, telephone （固定）など）。
* 10) day      :int64	    最終接触日の「日」。
* 11) month    :object	最終接触日の「月」。
* ※　duration削除 
* 13) campaign :int64     このキャンペーン（勧誘）期間中の接触回数。
* 14) pdays    :int64	    前回のキャンペーン（勧誘）接触から経過した日数（-1は未接触）。
* 15) previous :int64	    このキャンペーン（勧誘）以前の接触回数。
* 16) poutcome :object	前回のキャンペーン（勧誘）の成果（success(契約してくれた), failure(断られた) など）。
<br>
<br>
新しいカラム<br>

* 17) total_loan_count：int64　housingとloanの「yes」の合計数（負債の重さ）。
* 18) balance_per_age：float64　年齢に対する残高の比率
* 19) is_new_prospect：int64　previousが0かつpdaysが-1の完全新規フラグ。
* 20) day_month_interaction：int64　月と日の組み合わせによるボーナス・決算期特定

# ライブラリ

In [121]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# train.csvデータ加工

In [122]:
df = pd.read_csv('data/train.csv')

#### 17)  df["total_loan_count"]
[多重でローンしている人]<br>
0:借りていない<br>
1:個人ローン or 住宅ローンどちらかを借りている<br>
2:両方とも借りている <br>

In [123]:
df["total_loan_count"] = (df['housing'] == 'yes').astype(int) + (df['loan'] == 'yes').astype(int)
df["total_loan_count"].value_counts()

total_loan_count
1    384545
0    299595
2     65860
Name: count, dtype: int64

#### 18) df["balance_per_age"]
[年齢に対する残高の比率]<br>
1歳あたりいくら貯金しているか?
20歳の100万円と60歳の100万円を、モデルが同じ「100万円」として扱わないようにする<br>
※「35.1（上位25%）」より高ければ1、それ以外は0

In [124]:
df['balance_per_age'] = df['balance'] / df['age']
df['balance_per_age']

0          0.166667
1         13.526316
2         16.722222
3          1.259259
4         34.192308
            ...    
749995    44.206897
749996     9.144928
749997     4.340000
749998    -8.562500
749999    37.119048
Name: balance_per_age, Length: 750000, dtype: float64

In [125]:
df["balance_per_age"].describe() #四分位

count    750000.000000
mean         30.348067
std          70.209620
min        -348.652174
25%           0.000000
50%          16.121212
75%          35.148148
max        3645.074074
Name: balance_per_age, dtype: float64

In [126]:
df["balance_per_age"] = (df['balance'] / df['age'] > 35.1).astype(int)

In [127]:
df["balance_per_age"].value_counts()

balance_per_age
0    562309
1    187691
Name: count, dtype: int64

#### 19) df["is_new_prospect"]
本当の意味での新規客
previous == 0：今回のキャンペーンより前に、一度も接触がない。<br>
かつ<br>
pdays == -1：過去に接触した記録（日数）が全く存在しない

In [128]:
df['is_new_prospect'] = ((df['previous'] == 0) & (df['pdays'] == -1)).astype(int)
df["is_new_prospect"].value_counts()

is_new_prospect
1    672417
0     77583
Name: count, dtype: int64

#### 20) df["day_month_interaction"]
[四半期末かつ20日以降]<br>

In [129]:
df["day_month_interaction"] = ((df['month'].isin(['mar', 'jun', 'sep', 'dec'])) & (df['day'] >= 20)).astype(int)
df["day_month_interaction"].value_counts()

day_month_interaction
0    729495
1     20505
Name: count, dtype: int64

### 最終

In [130]:
df = df.drop(columns="duration")
df

,id,age,job,marital,education,default,balance,housing,loan,contact,...,month,campaign,pdays,previous,poutcome,y,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction
0,0,42,technician,married,secondary,no,7,no,no,cellular,...,aug,3,-1,0,unknown,0,0,0,1,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,...,jun,1,-1,0,unknown,0,0,0,1,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,...,may,2,-1,0,unknown,0,1,0,1,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,...,may,2,-1,0,unknown,0,1,0,1,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,...,feb,1,-1,0,unknown,1,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
749995,749995,29,services,single,secondary,no,1282,no,yes,unknown,...,jul,2,-1,0,unknown,1,1,1,1,0
749996,749996,69,retired,divorced,tertiary,no,631,no,no,cellular,...,aug,1,-1,0,unknown,0,0,0,1,0
749997,749997,50,blue-collar,married,secondary,no,217,yes,no,cellular,...,apr,1,-1,0,unknown,0,1,0,1,0
749998,749998,32,technician,married,secondary,no,-274,no,no,cellular,...,aug,6,-1,0,unknown,0,0,0,1,0


In [131]:
df.to_pickle("data.pkl")

# test.csvデータ加工

In [132]:
df_test = pd.read_csv("data/test.csv")

#### 17)  df_test["total_loan_count"]

In [133]:
df_test["total_loan_count"] = (df_test['housing'] == 'yes').astype(int) + (df_test['loan'] == 'yes').astype(int)
df_test["total_loan_count"].value_counts()

total_loan_count
1    127659
0    100382
2     21959
Name: count, dtype: int64

#### 18) df_test["balance_per_age"]

In [134]:
df_test['balance_per_age'] = df_test['balance'] / df_test['age']

In [135]:
df_test["balance_per_age"] = (df_test['balance'] / df_test['age'] > 35.1).astype(int)

In [136]:
df_test["balance_per_age"].value_counts()

balance_per_age
0    187547
1     62453
Name: count, dtype: int64

#### 19) df_test["is_new_prospect"]

In [137]:
df_test['is_new_prospect'] = ((df_test['previous'] == 0) & (df_test['pdays'] == -1)).astype(int)
df_test["is_new_prospect"].value_counts()

is_new_prospect
1    224106
0     25894
Name: count, dtype: int64

#### 20) df_test["day_month_interaction"]

In [138]:
df_test["day_month_interaction"] = ((df_test['month'].isin(['mar', 'jun', 'sep', 'dec'])) & (df_test['day'] >= 20)).astype(int)
df_test["day_month_interaction"].value_counts()

day_month_interaction
0    243042
1      6958
Name: count, dtype: int64

### 最終

In [139]:
df_test = df_test.drop(columns="duration")
df_test

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,campaign,pdays,previous,poutcome,total_loan_count,balance_per_age,is_new_prospect,day_month_interaction
0,750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,21,may,1,-1,0,unknown,1,1,1,0
1,750001,44,management,married,tertiary,no,23,yes,no,cellular,3,apr,2,-1,0,unknown,1,0,1,0
2,750002,36,self-employed,married,primary,no,46,yes,yes,cellular,13,may,2,-1,0,unknown,2,0,1,0
3,750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,29,may,1,-1,0,unknown,2,0,1,0
4,750004,28,technician,single,secondary,no,1950,yes,no,cellular,22,jul,1,-1,0,unknown,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,999995,43,management,married,tertiary,no,0,yes,no,cellular,18,nov,2,-1,0,unknown,1,0,1,0
249996,999996,40,services,married,unknown,no,522,yes,no,cellular,19,nov,1,189,1,failure,1,0,0,0
249997,999997,63,retired,married,primary,no,33,no,no,cellular,3,jul,1,92,8,success,0,0,0,0
249998,999998,50,blue-collar,married,primary,no,2629,yes,no,unknown,30,may,2,-1,0,unknown,1,1,1,0


In [141]:
df_test.to_pickle("test.pkl")